# **Initialization**

In [2]:
print('Start')

Start


In [3]:
# =========================================================
# 1. IMPORTS (Strictly included as requested)
# =========================================================
%load_ext autoreload
%autoreload 2

import os
import sys
import re
import gc
import math
import random
import time as pytime
import contextlib
import csv
import glob
from functools import lru_cache

# --- Third-Party Imports ---
import numpy as np
import pandas as pd
import pulp
import vrplib
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
from ortools.linear_solver import pywraplp

# --- Library Setup (Adjust path if necessary) ---
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\Evolutionary_algorithm"

if LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)
    print(f"📂 Library path added: {LIBRARY_PARENT_PATH}")

try:
    import evolutionary_algorithm_lib
    from evolutionary_algorithm_lib import (
        compile_chromosome_to_useable_function,
        combining_modified_didppy_solver_with_chromosome,
        evolution_algorithm_execution, # Explicitly importing execution function
        EAHyperparameters # Explicitly importing params class
    )
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    print("✅ Success! 'evolutionary_algorithm_lib' is imported and ready.")
except ImportError as e:
    print(f"❌ Error: Could not import 'evolutionary_algorithm_lib'.\nDetails: {e}")

📂 Library path added: C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\Evolutionary_algorithm
✅ Success! 'evolutionary_algorithm_lib' is imported and ready.


# **Data**

In [4]:
def read_formated_data(file_path):
    """
    Reads a Solomon format .txt file using vrplib and returns a dictionary 
    formatted for CVRPTW LP Relaxation and DIDP models.
    
    Ensures all numerical data (demand, time windows, service times, costs, capacity) 
    are returned as floats.
    """
    # 1. Read instance using vrplib
    # instance_format='solomon' ensures correct parsing of sections
    instance = vrplib.read_instance(file_path, instance_format='solomon')

    # 2. Extract Data & Cast to Float
    # 'edge_weight' is the distance matrix computed by vrplib
    travel_cost = instance['edge_weight'].astype(float).tolist()
    
    # 'node_coord' is available if you ever need it, but we use the pre-calc weights
    num_locations = len(instance['node_coord'])

    # 3. Return Bundle
    return {
        'num_locations': num_locations,
        'num_vehicles': int(instance.get('vehicles', 25)), 
        'capacity': float(instance['capacity']),
        
        # Cast demand to float list
        'demand': instance['demand'].astype(float).tolist(),
        
        # Cast Time Windows to float list
        # Col 0 is ready_time (earliest arrival), Col 1 is due_date (latest arrival)
        'ready_time': instance['time_window'][:, 0].astype(float).tolist(),
        'due_date': instance['time_window'][:, 1].astype(float).tolist(),
        
        # Cast Service Time to float list
        'service_time': instance['service_time'].astype(float).tolist(),
        
        # Use the pre-computed edge weights from vrplib
        'travel_cost': travel_cost
    }

# 1. Define your directory path
base_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\Solomon"

# 2. Construct the full file path
instance_path = os.path.join(base_path, "C101.txt")

# 3. Read the data using your new function
data = read_formated_data(instance_path)

# 4. Extract all necessary variables
# These variable names match what is typically expected by your DIDP/LP models
current_num_locations = data['num_locations']
current_num_vehicles  = data['num_vehicles']
current_capacity      = data['capacity']
current_cust_demand   = data['demand']
current_avail_time    = data['ready_time']
current_due_date      = data['due_date']
current_serve_time  = data['service_time']
current_travel_cost   = data['travel_cost']

# --- Sanity Check / Verification ---
print(f"Extraction Complete for {os.path.basename(instance_path)}")
print(f"------------------------------------------------")
print(f"Num Locations: {current_num_locations}")
print(f"Num Vehicles:  {current_num_vehicles}")
print(f"Capacity:      {current_capacity}")
print(f"Dimensions Check:")
print(f"  - Demand:       {len(current_cust_demand)}")
print(f"  - Ready Time:   {len(current_avail_time)}")
print(f"  - Due Date:     {len(current_due_date)}")
print(f"  - Service Time: {len(current_serve_time)}")
print(f"  - Cost Matrix:  {len(current_travel_cost)}x{len(current_travel_cost[0])}")
print(f"Demand : {current_cust_demand}")
print(f"Ready time: {current_avail_time}")
print(f"Due date (b): {current_due_date}")
print(f"Service time (s): {current_serve_time}")

# Display raw dictionary if needed
# display(data)

Extraction Complete for C101.txt
------------------------------------------------
Num Locations: 101
Num Vehicles:  25
Capacity:      200.0
Dimensions Check:
  - Demand:       101
  - Ready Time:   101
  - Due Date:     101
  - Service Time: 101
  - Cost Matrix:  101x101
Demand : [0.0, 10.0, 30.0, 10.0, 10.0, 10.0, 20.0, 20.0, 20.0, 10.0, 10.0, 10.0, 20.0, 30.0, 10.0, 40.0, 40.0, 20.0, 20.0, 10.0, 10.0, 20.0, 20.0, 10.0, 10.0, 40.0, 10.0, 10.0, 20.0, 10.0, 10.0, 20.0, 30.0, 40.0, 20.0, 10.0, 10.0, 20.0, 30.0, 20.0, 10.0, 10.0, 20.0, 10.0, 10.0, 10.0, 30.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 20.0, 40.0, 10.0, 30.0, 40.0, 30.0, 10.0, 20.0, 10.0, 20.0, 50.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 30.0, 20.0, 10.0, 10.0, 50.0, 20.0, 10.0, 10.0, 20.0, 10.0, 10.0, 30.0, 20.0, 10.0, 20.0, 30.0, 10.0, 20.0, 30.0, 10.0, 10.0, 10.0, 20.0, 40.0, 10.0, 30.0, 10.0, 30.0, 20.0, 10.0, 20.0]
Ready time: [0.0, 912.0, 825.0, 65.0, 727.0, 15.0, 621.0, 170.0, 255.0, 534.0, 357.0, 448.0, 652.0, 30.0, 567.0, 

# **1. Model and dual bound declaration**

In [5]:
def creation_of_didp_model_function():
    num_locations = current_num_locations
    num_vehicles = current_num_vehicles
    q = current_capacity
    cust_demand = current_cust_demand
    avail_time = current_avail_time
    due_date = current_due_date
    serve_time = current_serve_time
    travel_cost = current_travel_cost
    
    # =====================================================================================
    # DIDP Model Definition
    # =====================================================================================
    model = m_dp.Model(float_cost=True)

    # Object types for customers/locations and vehicles
    customer = model.add_object_type(number=num_locations)
    vehicle = model.add_object_type(number=num_vehicles)

    # -------------------- State Variables --------------------
    # Set of unvisited customers
    unvisited_locations = model.add_set_var(object_type=customer, target=list(range(1, num_locations)))

    # Per-vehicle state variables, stored in Python lists for easy access
    vehicle_locations = [
    model.add_element_var(object_type=customer, target=0, name=f"loc_v{v}")
    for v in range(num_vehicles)
    ]
    vehicle_loads = [
    model.add_float_var(target=0, name=f"load_v{v}")
    for v in range(num_vehicles)
    ]
    vehicle_times = [
    model.add_float_resource_var(target=0, less_is_better=True, name=f"time_v{v}")
    for v in range(num_vehicles)
    ]
    chosen_customer = model.add_element_var(object_type=customer, target=0, name="chosen_customer")
    alpha = model.add_int_var(target=0, name="alpha")

    # -------------------- Tables of Constants --------------------
    demand = model.add_float_table(cust_demand)
    ready_time = model.add_float_table(avail_time)
    due_time = model.add_float_table(due_date)
    service_time = model.add_float_table(serve_time)
    travel_time = model.add_float_table(travel_cost)

    # -------------------- Transitions --------------------
    # Choose customer j to be visited next
    for j in range(1, num_locations):
        choosing_customer_transition = m_dp.Transition(
            name=f"choose_customer_{j}_to_visit",
            cost=m_dp.FloatExpr.state_cost(),
            preconditions =[
                unvisited_locations.contains(j),
                alpha == 0
                ],
            effects=[
                (chosen_customer, j),
                (alpha, 1)
                ],
        )
        model.add_transition(choosing_customer_transition)

        # Transition to visit a customer j with a vehicle v
    for v in range(num_vehicles):
        arrival_time = m_dp.max(
            vehicle_times[v] + travel_time[vehicle_locations[v], chosen_customer],
            ready_time[chosen_customer]
        )

        departure_time = arrival_time + service_time[chosen_customer]

        visit_transition = m_dp.Transition(
            name=f"visit_chosen_customer_with_vehicle_{v}",
            cost=travel_time[vehicle_locations[v], chosen_customer] + m_dp.FloatExpr.state_cost(),
            preconditions=[
                unvisited_locations.contains(chosen_customer),
                vehicle_loads[v] + demand[chosen_customer] <= q,
                arrival_time <= due_time[chosen_customer],
                alpha == 1,
            ],
            effects=[
                (unvisited_locations, unvisited_locations.remove(chosen_customer)),
                (vehicle_locations[v], chosen_customer),
                (vehicle_loads[v], vehicle_loads[v] + demand[chosen_customer]),
                (vehicle_times[v], departure_time),
                (alpha, 0),
            ],
        )

        model.add_transition(visit_transition)


    # Transitions for each vehicle to return to the depot after all customers are served
    for v in range(num_vehicles):
        return_to_depot_transition = m_dp.Transition(
            name=f"return_vehicle_{v}_to_depot",
            cost=travel_time[vehicle_locations[v], 0] + m_dp.FloatExpr.state_cost(),
            preconditions=[unvisited_locations.is_empty(), vehicle_locations[v] != 0],
            effects=[(vehicle_locations[v], 0)],
        )
        model.add_transition(return_to_depot_transition)

    # --- 1. Global Capacity Constraint (The Efficient "Cut") ---
    # Logic: Total Capacity Available >= Total Demand Remaining
    # Summing variables in Python creates a DIDP expression automatically
    total_current_load = sum(vehicle_loads) 
    total_fleet_capacity = num_vehicles * q
    
    # demand[unvisited_locations] automatically sums the weight of items in the set
    model.add_state_constr(
        (total_fleet_capacity - total_current_load) >= demand[unvisited_locations]
    )
    
    # -------------------- Base Case --------------------
    # All customers visited AND all vehicles are at the depot
    base_conditions = [unvisited_locations.is_empty()]
    for v in range(num_vehicles):
        base_conditions.append(vehicle_locations[v] == 0)
    model.add_base_case(base_conditions)
 
    # =========================================================
    # 3. Bundle Metadata
    # =========================================================
    metadata = {
        "unvisited_locations": unvisited_locations,
        "vehicle_locations": vehicle_locations,
        "vehicle_loads": vehicle_loads,
        "vehicle_times": vehicle_times,
        "distance_matrix": travel_cost,
        "demand": cust_demand,
        "due_time": due_date,
        "ready_time": avail_time,
        "service_time": serve_time,
        "capacity": q,
        "num_vehicles": num_vehicles,
        "num_locations": num_locations
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

In [6]:
def create_persistent_lp_relaxation_3_index_dual_bounds(metadata):
    """
    Creates a persistent 3-Index CVRP Relaxation with LRU CACHING.
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_locations']
    n_vehicles = metadata['num_vehicles']
    capacity = metadata['capacity']
    demands = metadata['demand']          
    dist_matrix = metadata['distance_matrix'] 
    
    # State Variables
    unvisited_var = metadata['unvisited_locations']
    vehicle_vars = metadata['vehicle_locations']

    # ==========================================
    # 1. INITIALIZATION (Runs Once)
    # ==========================================
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0

    infinity = solver.infinity()
    x = {}
    y = {}
    u = {}
    cons_assignment = {} 

    # --- Variables & Constraints Construction (Same as before) ---
    # 1. Create x_ijk (Flow)
    for k in range(n_vehicles):
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i != j:
                    x[(k, i, j)] = solver.NumVar(0, 1, f'x_{k}_{i}_{j}')

    # 2. Create y_ik (Assignment)
    for i in range(n_nodes):
        for k in range(n_vehicles):
            y[(i, k)] = solver.NumVar(0, 1, f'y_{i}_{k}')

    # 3. Create u_ik (Potentials)
    for i in range(1, n_nodes):
        for k in range(n_vehicles):
            u[(i, k)] = solver.NumVar(0, capacity, f'u_{i}_{k}')

    # (1.29) Customer Assignment
    for i in range(1, n_nodes):
        c = solver.Constraint(0, 0, f'assign_{i}')
        for k in range(n_vehicles):
            c.SetCoefficient(y[(i, k)], 1)
        cons_assignment[i] = c

    # (1.30) Depot Usage
    c_depot = solver.Constraint(0, n_vehicles, 'depot_usage')
    for k in range(n_vehicles):
        c_depot.SetCoefficient(y[(0, k)], 1)

    # (1.31) Flow Conservation
    for k in range(n_vehicles):
        for i in range(n_nodes):
            c_out = solver.Constraint(0, 0, f'flow_out_{i}_{k}')
            c_out.SetCoefficient(y[(i, k)], -1)
            for j in range(n_nodes):
                if i != j: c_out.SetCoefficient(x[(k, i, j)], 1)
            
            c_in = solver.Constraint(0, 0, f'flow_in_{i}_{k}')
            c_in.SetCoefficient(y[(i, k)], -1)
            for j in range(n_nodes):
                if i != j: c_in.SetCoefficient(x[(k, j, i)], 1)

    # (1.37) & (1.38) MTZ
    for k in range(n_vehicles):
        for i in range(1, n_nodes):
            for j in range(1, n_nodes):
                if i != j:
                    c = solver.Constraint(-infinity, float(capacity - demands[j]), f'mtz_{k}_{i}_{j}')
                    c.SetCoefficient(u[(i, k)], 1)
                    c.SetCoefficient(u[(j, k)], -1)
                    c.SetCoefficient(x[(k, i, j)], capacity)
    
    # --- Objective ---
    objective = solver.Objective()
    for k in range(n_vehicles):
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i != j:
                    objective.SetCoefficient(x[(k, i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # ==========================================
    # 2. CACHED SOLVER WORKER
    # ==========================================
    # We define the solver logic as an internal function and cache IT.
    # The key will be a tuple of active nodes.
    
    @lru_cache(maxsize=10000)
    def _solve_cached_lp(active_nodes_tuple):
        """
        Internal worker that sets bounds and solves.
        Input must be a hashable tuple (sorted list of active nodes).
        """
        # Convert tuple back to set for fast lookup
        active_set = set(active_nodes_tuple)

        # Update Constraints
        for i in range(1, n_nodes):
            if i in active_set:
                # ACTIVE
                cons_assignment[i].SetBounds(1, 1)
                for k in range(n_vehicles):
                    u[(i, k)].SetBounds(demands[i], capacity)
            else:
                # INACTIVE
                cons_assignment[i].SetBounds(0, 0)
                for k in range(n_vehicles):
                    u[(i, k)].SetBounds(0, 0)

        # Solve with Time Limit
        solver.SetTimeLimit(100) # 100ms limit
        status = solver.Solve()
        
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # ==========================================
    # 3. HEURISTIC WRAPPER
    # ==========================================
    def h_lp_relaxation_3_idx(state):
        unvisited = state[unvisited_var]
        current_locs = [state[v_var] for v_var in vehicle_vars]
        
        # Quick exit
        if not unvisited and all(loc == 0 for loc in current_locs): 
            return 0.0

        # Build Active Set
        # Valid nodes are Unvisited U Current_Vehicle_Locations
        active_customers = set(unvisited)
        for loc in current_locs:
            if loc != 0:
                active_customers.add(loc)
        
        # CRITICAL: Convert to sorted tuple for the cache key
        # If we passed the set or list directly, lru_cache would fail (unhashable)
        # Sorting ensures that visiting {1, 2} is the same cache hit as visiting {2, 1}
        active_key = tuple(sorted(list(active_customers)))
        
        return _solve_cached_lp(active_key)

    return h_lp_relaxation_3_idx

def create_persistent_lp_relaxation_2_index_dual_bounds(metadata):
    """
    Creates a persistent 2-Index (Flow-based) CVRP Relaxation with LRU CACHING.
    - SCALE: Capable of handling N=100 in < 0.1s per node.
    - LOGIC: Drops 'Vehicle' dimension. Treats flow as aggregate.
    - SOLVER: Google OR-Tools (GLOP).
    """
    # --- Extract Static Data (Matched to DIDP Metadata) ---
    n_nodes = metadata['num_locations']
    n_vehicles = metadata['num_vehicles']
    capacity = metadata['capacity']
    
    # Note: These are DIDP Table objects or Lists
    demands = metadata['demand']
    dist_matrix = metadata['distance_matrix']
    
    # State Variables
    unvisited_var = metadata['unvisited_locations']
    vehicle_vars = metadata['vehicle_locations'] # List of vehicle element variables

    # ==========================================
    # 1. INITIALIZATION (Runs Once)
    # ==========================================
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0

    infinity = solver.infinity()

    # --- Variables ---
    # x[i, j]: Binary flow from i to j (Aggregated over all vehicles)
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    # u[i]: Accumulated load variable for MTZ
    u = {i: solver.NumVar(0, capacity, f'u_{i}') for i in range(1, n_nodes)}

    # --- Constraints ---
    cons_degree_out = {} # Outgoing degree 
    cons_degree_in = {}  # Incoming degree

    # 1. Degree Constraints (Customers 1..N)
    # Each active customer must have exactly 1 outgoing and 1 incoming edge
    for i in range(1, n_nodes):
        # Outgoing
        c_out = solver.Constraint(0, 0, f'deg_out_{i}')
        for j in range(n_nodes):
            if i != j:
                c_out.SetCoefficient(x[(i, j)], 1)
        cons_degree_out[i] = c_out

        # Incoming
        c_in = solver.Constraint(0, 0, f'deg_in_{i}')
        for j in range(n_nodes):
            if i != j:
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_degree_in[i] = c_in

    # 2. Depot Degree Constraints (Static)
    # Total outgoing flow from Depot == Number of active vehicles (<= K)
    # We relax this to <= K for lower bound purposes
    c_depot_out = solver.Constraint(0, n_vehicles, 'depot_out')
    for j in range(1, n_nodes):
        c_depot_out.SetCoefficient(x[(0, j)], 1)
    
    # Total incoming flow to Depot <= K
    c_depot_in = solver.Constraint(0, n_vehicles, 'depot_in')
    for i in range(1, n_nodes):
        c_depot_in.SetCoefficient(x[(i, 0)], 1)

    # 3. MTZ / Capacity Constraints
    # Standard MTZ: u_j - u_i + Capacity * x_ij <= Capacity - d_j
    # This prevents subtours and ensures capacity compliance roughly.
    for i in range(1, n_nodes):
        for j in range(1, n_nodes):
            if i != j:
                # [Correction] Explicit float cast for RHS to avoid numpy int errors
                rhs = float(capacity - demands[j])
                c = solver.Constraint(-infinity, rhs, f'mtz_{i}_{j}')
                
                c.SetCoefficient(u[j], 1)
                c.SetCoefficient(u[i], -1)
                c.SetCoefficient(x[(i, j)], capacity)

    # --- Objective ---
    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # ==========================================
    # 2. CACHED SOLVER WORKER (Internal)
    # ==========================================
    # We define the solver logic as an internal function and cache IT.
    # The key will be a tuple of active nodes.
    
    @lru_cache(maxsize=10000)
    def _solve_cached_lp_2_idx(active_nodes_tuple):
        """
        Internal worker that sets bounds and solves.
        Input must be a hashable tuple (sorted list of active nodes).
        """
        # Convert tuple back to set for fast lookup
        active_customers = set(active_nodes_tuple)

        # Toggle Active Nodes
        # We iterate 1..N to update bounds based on current state
        for i in range(1, n_nodes):
            if i in active_customers:
                # ACTIVE:
                # Degree must be 1 (Visited exactly once)
                cons_degree_out[i].SetBounds(1, 1)
                cons_degree_in[i].SetBounds(1, 1)
                # Load variable active
                u[i].SetBounds(demands[i], capacity)
            else:
                # INACTIVE:
                # Degree must be 0 (Removed from graph)
                cons_degree_out[i].SetBounds(0, 0)
                cons_degree_in[i].SetBounds(0, 0)
                # Load variable forced to 0
                u[i].SetBounds(0, 0)
        
        # Solve
        solver.SetTimeLimit(100) # 100ms
        status = solver.Solve()
        
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # ==========================================
    # 3. HEURISTIC WRAPPER
    # ==========================================
    def h_lp_relaxation_2_idx(state):
        unvisited = state[unvisited_var]
        
        # [UPDATED]: Retrieve locations for ALL vehicles
        current_locs = [state[v_var] for v_var in vehicle_vars]

        # Optimization: Solved state
        if not unvisited and all(loc == 0 for loc in current_locs): 
            return 0.0

        # Define Active Set: Unvisited
        active_customers = set(unvisited)
        
        # [UPDATED]: Add current locations to active set (treating them as nodes to be covered)
        # This provides a valid relaxation for the remaining tour cost
        for loc in current_locs:
            if loc != 0:
                active_customers.add(loc)

        # CRITICAL: Convert to sorted tuple for the cache key
        # Sorting ensures that visiting {1, 2} is the same cache hit as visiting {2, 1}
        active_key = tuple(sorted(list(active_customers)))

        return _solve_cached_lp_2_idx(active_key)

    return h_lp_relaxation_2_idx

def create_persistent_cvrptw_cordeau_relaxed_model(metadata):
    """
    Creates a persistent 3-Index CVRPTW Relaxation based on Cordeau (2002).
    - Includes Time Window variables (w) and Capacity constraints.
    - Uses Google OR-Tools (GLOP) with LRU Caching.
    """
    # --- Extract Static Data ---
    num_locations = metadata['num_locations'] # Original N
    num_vehicles = metadata['num_vehicles']
    capacity = metadata['capacity']
    
    # Original Data
    cust_demand = metadata['demand']
    serve_time = metadata['service_time']
    ready_time = metadata['ready_time']
    due_date = metadata['due_time']
    travel_cost = metadata['distance_matrix']

    # State Variables
    unvisited_var = metadata['unvisited_locations']
    vehicle_vars = metadata['vehicle_locations']

    # --- 0. Data Augmentation (StartDepot=0, EndDepot=N) ---
    StartDepot = 0
    EndDepot = num_locations 
    
    # Augment lists (Add dummy end node)
    aug_demand = cust_demand + [0.0]
    aug_serve = serve_time + [0.0]
    aug_ready = ready_time + [ready_time[0]]
    aug_due = due_date + [due_date[0]]
    
    # Augment Cost Matrix
    aug_cost = [row[:] + [row[0]] for row in travel_cost] 
    aug_cost.append(aug_cost[0][:]) 

    # Sets
    K = range(num_vehicles)
    N_customers = range(1, num_locations) # 1..N-1
    V_all = range(num_locations + 1)      # 0..N
    
    # --- FIX: Use a SET for A to ensure O(1) lookups ---
    A = set()
    for i in V_all:
        for j in V_all:
            if i == j: continue
            if i == EndDepot: continue   # Nothing leaves EndDepot
            if j == StartDepot: continue # Nothing enters StartDepot
            A.add((i,j))

    # Helper for Big-M
    #big_m = {}
    #for (i, j) in A:
        #val = aug_due[i] + aug_serve[i] + aug_cost[i][j] - aug_ready[j]
    #   big_m[(i,j)] = 10**7 # max(val, 0)

    # ==========================================
    # 1. INITIALIZATION (Runs Once)
    # ==========================================
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0
    
    infinity = solver.infinity()

    # --- Variables ---
    x = {} # Flow x_k_i_j
    w = {} # Time w_k_i

    # Note: iterating over set A is fast
    for k in K:
        for (i, j) in A:
            x[(k, i, j)] = solver.NumVar(0, 1, f"x_{k}_{i}_{j}")
        for i in V_all:
            w[(k, i)] = solver.NumVar(0, 10**6, f"w_{k}_{i}")

    # --- Constraints ---
    
    # 1. Assignment Constraints (Mutable)
    cons_assignment = {}
    for i in N_customers:
        c = solver.Constraint(0, 0, f"assign_{i}")
        for k in K:
            # We iterate V_all and check membership in A (O(1) now)
            for j in V_all:
                if (i, j) in A:
                    c.SetCoefficient(x[(k, i, j)], 1)
        cons_assignment[i] = c

    # 2. Depot Departure
    for k in K:
        c = solver.Constraint(1, 1, f"depot_out_{k}")
        for j in V_all:
            if (StartDepot, j) in A:
                c.SetCoefficient(x[(k, StartDepot, j)], 1)

    # 3. Flow Conservation
    for k in K:
        for j in N_customers:
            c = solver.Constraint(0, 0, f"flow_{k}_{j}")
            # Inflow
            for i in V_all:
                if (i, j) in A: c.SetCoefficient(x[(k, i, j)], 1)
            # Outflow
            for i in V_all:
                if (j, i) in A: c.SetCoefficient(x[(k, j, i)], -1)

    # 4. Depot Arrival
    for k in K:
        c = solver.Constraint(1, 1, f"depot_in_{k}")
        for i in V_all:
            if (i, EndDepot) in A:
                c.SetCoefficient(x[(k, i, EndDepot)], 1)

    # 5. Capacity
    for k in K:
        c = solver.Constraint(-infinity, capacity, f"cap_{k}")
        for i in N_customers:
            for j in V_all:
                if (i, j) in A:
                    c.SetCoefficient(x[(k, i, j)], aug_demand[i])

    # 6. Time Propagation (Linearized)
    for k in K:
        for (i, j) in A:
            M = 10**6
            rhs = M - aug_serve[i] - aug_cost[i][j]
            c = solver.Constraint(-infinity, rhs, f"time_prop_{k}_{i}_{j}")
            c.SetCoefficient(w[(k, i)], 1)
            c.SetCoefficient(w[(k, j)], -1)
            c.SetCoefficient(x[(k, i, j)], M)

    # 7. Time Windows (Linked to Visits)
    for k in K:
        for i in N_customers:
            # Visit variable (sum outgoing)
            visit_expr = [] 
            for j in V_all:
                if (i, j) in A: visit_expr.append(x[(k, i, j)])
            
            # Lower Bound: w >= a * visit
            c_lb = solver.Constraint(-infinity, 0, f"tw_lb_{k}_{i}")
            c_lb.SetCoefficient(w[(k, i)], -1)
            for var in visit_expr: c_lb.SetCoefficient(var, aug_ready[i])

            # Upper Bound: w <= b * visit
            c_ub = solver.Constraint(-infinity, 0, f"tw_ub_{k}_{i}")
            c_ub.SetCoefficient(w[(k, i)], 1)
            for var in visit_expr: c_ub.SetCoefficient(var, -aug_due[i])

    # --- Objective ---
    objective = solver.Objective()
    for k in K:
        for (i, j) in A:
            objective.SetCoefficient(x[(k, i, j)], aug_cost[i][j])
    objective.SetMinimization()

    # ==========================================
    # 2. CACHED SOLVER WORKER
    # ==========================================
    @lru_cache(maxsize=10000)
    def _solve_cordeau(active_tuple):
        """
        Internal worker to solve the Cordeau relaxation for a specific set of active nodes.
        active_tuple: Sorted tuple of indices that MUST be visited.
        """
        active_set = set(active_tuple)

        # Update Assignment Constraints
        for i in N_customers:
            if i in active_set:
                # Must be visited exactly once
                cons_assignment[i].SetBounds(1, 1)
            else:
                # Already visited / Inactive -> Force flow to 0 (remove from graph)
                cons_assignment[i].SetBounds(0, 0)

        # Solve
        solver.SetTimeLimit(100) # 100ms limit per call
        status = solver.Solve()

        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # ==========================================
    # 3. HEURISTIC WRAPPER
    # ==========================================
    def h_lp_cordeau(state):
        unvisited = state[unvisited_var]
        
        # If no unvisited customers, cost is 0 (or cost to return to depot)
        if not unvisited:
            return 0.0

        # Create cache key: Sorted tuple of unvisited nodes
        active_key = tuple(sorted(list(unvisited)))
        
        return _solve_cordeau(active_key)

    return h_lp_cordeau

In [7]:
def dual_bound_expression_function(didp_bundle):
    """ 
    Returns a dictionary of heuristic functions (bounds) bound to the model data.
    Includes LP relaxations, Flow, MST, 1-Tree, Assignment, and Eigenvalue bounds.
    """
    
    model, metadata = didp_bundle
    
    # --- 1. Extract DIDP Variables (from Metadata) ---
    unvisited_var = metadata['unvisited_locations']
    vehicle_vars = metadata['vehicle_locations']  # List of ElementVars
    
    capacity = metadata['capacity']
    num_vehicles = metadata['num_vehicles']
    n_nodes = metadata['num_locations'] 

    # --- 2. Extract RAW Data for Heuristics ---
    # Metadata contains DIDP Tables (symbolic). Heuristics (MST, Eigen, etc.) need 
    # the actual numerical lists/matrices. We fetch them from global scope if available.
    
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list) 
    demand_list = metadata['demand']

    # --- 3. Pre-computation for Bounds (Run once per problem) ---
    # masked_cost: diagonal is infinity to ignore self-loops
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    
    # min_outgoing_arr[i] = min cost to leave node i
    min_outgoing_arr = np.min(masked_cost, axis=1)
    
    # min_incoming_arr[j] = min cost to enter node j
    min_incoming_arr = np.min(masked_cost, axis=0)

    # ==========================================
    # 4. LP Relaxation Bounds (Persistent)
    # ==========================================
    # Pass metadata through (The LP functions must expect the specific keys you defined)
    h_lp_relaxation_3_idx = create_persistent_lp_relaxation_3_index_dual_bounds(metadata=metadata)
    h_lp_relaxation_2_idx = create_persistent_lp_relaxation_2_index_dual_bounds(metadata=metadata)
    h_lp_cordeau = create_persistent_cvrptw_cordeau_relaxed_model(metadata=metadata)
    
    # ==========================================
    # 5. Define Combinatorial Bounds (OPTIMIZED WITH LRU CACHE)
    # =========================================

    # --- Flow Bound ---
    # Although h_flow is fast (O(N)), caching avoids repeated list comprehensions.
    @lru_cache(maxsize=100000)
    def _cached_flow_calc(unvisited_tuple):
        # Sum demand of unvisited * dist to depot
        s = sum(demand_list[v] * distance_list[0][v] for v in unvisited_tuple)
        return float(round((2.0 / capacity) * s))

    def h_flow(state):
        U = state[unvisited_var]
        if not U: return 0.0
        try:
            # Convert to tuple for cache
            U_tuple = tuple(sorted(list(U)))
            return _cached_flow_calc(U_tuple)
        except Exception as e:
            print(f"Error in h_flow: {e}")
            return 0.0

    # --- Degree Average Bound (Local Subgraph) ---
    # This involves matrix slicing and min/sum operations. 
    # Since it depends on vehicle locations, we cache based on the "Active Node Set".
    @lru_cache(maxsize=100000)
    def _cached_degree_average_calc(active_nodes_tuple):
        nodes = list(active_nodes_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)

        mins_in = np.min(sub_mat, axis=0) 
        mins_out = np.min(sub_mat, axis=1)
        
        sum_in = np.sum(mins_in)      
        sum_out = np.sum(mins_out)   
        return float(0.5 * (sum_in + sum_out))

    def h_degree_average(state):
        U = state[unvisited_var]
        curr_locs = [state[v_var] for v_var in vehicle_vars]

        if not U and all(loc == 0 for loc in curr_locs):
            return 0.0
        try: 
            # 1. Build the full set of active nodes (Unvisited + Vehicles + Depot)
            active_set = set(U)
            for loc in curr_locs:
                if loc != 0:
                    active_set.add(loc)
            active_set.add(0) # Always include depot
            
            # 2. Convert to sorted tuple for cache key
            active_tuple = tuple(sorted(list(active_set)))
            
            return _cached_degree_average_calc(active_tuple)
        except Exception as e:
            print(f"Error in h_degree_average: {e}")
            return 0.0

    # --- Global Min Flow Bound ---
    # Optimization: Cache the sum of unvisited nodes part. 
    # The vehicle part is fast and dynamic, so we add it outside.
    @lru_cache(maxsize=100000)
    def _cached_global_min_flow_unvisited_part(unvisited_tuple):
        val_out = sum(min_outgoing_arr[u] for u in unvisited_tuple)
        val_in = sum(min_incoming_arr[u] for u in unvisited_tuple)
        return val_out, val_in

    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr_locs = [state[v_var] for v_var in vehicle_vars]
        
        if not U and all(loc == 0 for loc in curr_locs):
            return 0.0
        try:
            # Get cached sums for the static unvisited part
            U_tuple = tuple(sorted(list(U)))
            val_out, val_in = _cached_global_min_flow_unvisited_part(U_tuple)

            # Add dynamic vehicle parts
            for loc in curr_locs:
                if loc != 0:
                    val_out += min_outgoing_arr[loc]
            
            # Logic check: At least one vehicle must return to depot
            if val_out > 0: 
                val_in += min_incoming_arr[0]
            
            return float(max(val_out, val_in))
        except Exception as e:
            print(f"Error in h_global_min_flow: {e}")
            return 0.0

    # --- MST Bound ---
    @lru_cache(maxsize=100000)
    def _cached_mst_calc(unvisited_tuple):
        if not unvisited_tuple: return 0.0
        nodes = [0] + sorted(list(unvisited_tuple))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    def h_mst(state):
        U = state[unvisited_var]
        # Convert to tuple so it can be hashed by lru_cache
        U_tuple = tuple(sorted(list(U)))
        return _cached_mst_calc(U_tuple)

    # --- 1-Tree Bound ---
    @lru_cache(maxsize=100000)
    def _cached_1tree_calc(unvisited_tuple):
        subset_nodes = list(unvisited_tuple) # Already sorted from wrapper
        
        # Cheapest 2 edges connected to depot from the subset
        depot_edges = sorted(cost_matrix[0, subset_nodes])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        
        if len(subset_nodes) > 1:
            sub_mat = cost_matrix[np.ix_(subset_nodes, subset_nodes)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0 
        return float(mst_val + e1 + e2)

    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        try:
            U_tuple = tuple(sorted(list(U)))
            return _cached_1tree_calc(U_tuple)
        except Exception as e:
            print(f"Error in h_1tree: {e}")
            return 0.0

    # --- Assignment Bound ---
    @lru_cache(maxsize=100000)
    def _cached_assignment_calc(unvisited_tuple):
        nodes = [0] + sorted(list(unvisited_tuple))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        
        # Prepare for linear assignment (diagonal inf)
        assign_mat = sub_mat.astype(float).copy()
        np.fill_diagonal(assign_mat, np.inf)
        
        row_ind, col_ind = linear_sum_assignment(assign_mat)
        return float(assign_mat[row_ind, col_ind].sum())

    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        try:
            U_tuple = tuple(sorted(list(U)))
            return _cached_assignment_calc(U_tuple)
        except Exception as e:
            print(f"Error in h_assignment: {e}")
            return 0.0

    # --- Eigenvalue Bound ---
    @lru_cache(maxsize=100000)
    def _cached_eigen_calc(unvisited_tuple):
        nodes = [0] + sorted(list(unvisited_tuple))
        N_sub = len(nodes)
        if N_sub < 2: return 0.0

        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N_sub, 1))
        P = np.eye(N_sub) - (one @ one.T) / N_sub
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
            
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N_sub) for k in range(1, N_sub)])

        phi = 0.0
        if N_sub > 1:
            if N_sub % 2 == 1:
                num_terms = (N_sub - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
            else:
                num_sum_terms = N_sub // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N_sub-2 < len(eigvals):
                        phi += 2 * eigvals[N_sub - 2]
                elif N_sub > 1 and N_sub-2 < len(eigvals):
                    phi = 2 * eigvals[N_sub - 2]
        return float(phi)

    def h_eigen(state):
        U = state[unvisited_var]
        try:
            U_tuple = tuple(sorted(list(U)))
            return _cached_eigen_calc(U_tuple)
        except Exception as e:
            print(f"Error in h_eigen: {e}")
            return 0.0

    # Return valid registry
    return automatic_creation_of_dual_bounds_registry(locals())

# --- Execution Line ---
# Ensure creation_of_didp_model_function is defined and globals exist
dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())
display(dual_bound_functions_registry)

{'h_lp_relaxation_3_idx': <function __main__.create_persistent_lp_relaxation_3_index_dual_bounds.<locals>.h_lp_relaxation_3_idx(state)>,
 'h_lp_relaxation_2_idx': <function __main__.create_persistent_lp_relaxation_2_index_dual_bounds.<locals>.h_lp_relaxation_2_idx(state)>,
 'h_lp_cordeau': <function __main__.create_persistent_cvrptw_cordeau_relaxed_model.<locals>.h_lp_cordeau(state)>,
 'h_flow': <function __main__.dual_bound_expression_function.<locals>.h_flow(state)>,
 'h_degree_average': <function __main__.dual_bound_expression_function.<locals>.h_degree_average(state)>,
 'h_global_min_flow': <function __main__.dual_bound_expression_function.<locals>.h_global_min_flow(state)>,
 'h_mst': <function __main__.dual_bound_expression_function.<locals>.h_mst(state)>,
 'h_1tree': <function __main__.dual_bound_expression_function.<locals>.h_1tree(state)>,
 'h_assignment': <function __main__.dual_bound_expression_function.<locals>.h_assignment(state)>,
 'h_eigen': <function __main__.dual_bound_

# **Hyperparameters**

In [8]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 15       # Size of the population in each generation
GENERATIONS = 10       # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.10
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 10               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=random.randint(2, 10)
tournament_probability=0.8
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
OPTIMAL_COST_REFERENCE= 0
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 10 #seconds

# **Execution**

In [ ]:
# =========================================================
# 1. BATCH CONFIGURATION
# =========================================================
SEARCH_DIRS = [
    r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\Solomon",
    r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\homberger_200_customer_instances",
]

INPUT_CSV = "CVRPTW_single_dual_bound_results_10s_lim.csv"
OUTPUT_CSV = "result_CVRPTW_EA_dual_bound.csv"
LOGS_DIR = "CVRPTW_EA_Batch_Logs"

if not os.path.exists(LOGS_DIR):
    os.makedirs(LOGS_DIR)

# =========================================================
# 2. HELPER UTILITIES
# =========================================================
def update_globals_for_cvrptw(file_path):
    """Calls read_formated_data and updates global variables."""
    global current_num_locations, current_num_vehicles, current_capacity
    global current_cust_demand, current_travel_cost
    global current_avail_time, current_due_date, current_serve_time
    
    data = read_formated_data(file_path)
    
    current_num_locations = data['num_locations']
    current_num_vehicles  = data['num_vehicles']
    current_capacity      = data['capacity']
    current_cust_demand   = data['demand']
    current_travel_cost   = data['travel_cost']
    current_avail_time    = data['ready_time']
    current_due_date      = data['due_date']
    current_serve_time    = data['service_time']
    return True

def find_file_in_dirs(instance_name, search_dirs):
    for folder in search_dirs:
        path = os.path.join(folder, instance_name)
        if os.path.exists(path): return path
        if not instance_name.endswith('.txt'):
            path_txt = os.path.join(folder, instance_name + ".txt")
            if os.path.exists(path_txt): return path_txt
    return None

def get_clean_log_name(instance_name):
    """Removes extension from instance name to create clean log name."""
    # E.g., "C101.txt" -> "C101" -> "C101_log.txt"
    base_name = os.path.splitext(instance_name)[0]
    return f"{base_name}_log.txt"

def get_processed_instances(csv_path, logs_dir):
    """Returns instances found in BOTH the output CSV and logs directory."""
    csv_instances = set()
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if 'Instance' in df.columns:
                csv_instances = set(df['Instance'].unique())
        except: pass 

    # Check logs using the clean name format
    final_processed = set()
    for inst in csv_instances:
        expected_log = get_clean_log_name(inst)
        if os.path.exists(os.path.join(logs_dir, expected_log)):
            final_processed.add(inst)
            
    return final_processed

def clean_batch_data(csv_path, logs_dir):
    """Ensures consistency between CSV and Logs."""
    print("🧹 Starting Data Cleanup...")
    if not os.path.exists(csv_path):
        print("   -> CSV not found. Nothing to clean.")
        return

    try:
        df = pd.read_csv(csv_path)
    except pd.errors.EmptyDataError:
        return

    original_len = len(df)
    if 'Instance' in df.columns:
        df.drop_duplicates(subset=['Instance'], keep='last', inplace=True)
        
        valid_indices = []
        for index, row in df.iterrows():
            inst = row['Instance']
            # [FIX] Check for the clean log name
            expected_log = os.path.join(logs_dir, get_clean_log_name(inst))
            if os.path.exists(expected_log):
                valid_indices.append(index)
        
        df = df.loc[valid_indices]
        df.to_csv(csv_path, index=False)
        print(f"   -> Cleanup finished. Rows: {len(df)} (was {original_len}).")
    print("✨ Cleanup Complete.\n")

# =========================================================
# 3. MAIN BATCH EXECUTION LOOP
# =========================================================

# A. Prepare
clean_batch_data(OUTPUT_CSV, LOGS_DIR)
df_input = pd.read_csv(INPUT_CSV)
processed_instances = get_processed_instances(OUTPUT_CSV, LOGS_DIR)

print(f"🚀 MODE: BATCH RUN DETECTED (Source: {INPUT_CSV})")
if len(processed_instances) < len(df_input):
    print(f"   -> Resuming: {len(processed_instances)}/{len(df_input)} instances already completed.")
else:
    print(f"   -> All {len(processed_instances)} instances completed. No further processing needed.")

# B. Iterate
for index, row in df_input.iterrows():
    instance_name = row['Instance']
    
    # [FIX] Generate the clean log path here
    log_filename = get_clean_log_name(instance_name)
    log_file_path = os.path.join(LOGS_DIR, log_filename)

    # 1. Resume Check
    if instance_name in processed_instances and os.path.exists(log_file_path):
        continue

    # 2. Get Reference Cost
    optimal_cost =  row.get('Best Known Cost', 0)
    
    # 3. Find File
    file_path = find_file_in_dirs(instance_name, SEARCH_DIRS)
    if not file_path:
        print(f"\n⚠️ Skipping {instance_name}: File not found in search directories.")
        continue

    print(f"\nProcessing {instance_name} (Ref Cost: {optimal_cost})...")

    try:
        # 4. Update Global Data
        update_globals_for_cvrptw(file_path)

        # 5. Configure Params
        params = EAHyperparameters(
            # --- 1. Population ---
            population_size=POPULATION_SIZE,          
            generations=GENERATIONS,
            crossover_rate=CROSSOVER_RATE,
            mutation_rate=MUTATION_RATE,
            elitism_rate=ELITISM_RATE,           

            # --- 2. Ranges & Constraints ---
            lb_range_of_constant=LB_range_of_constant,
            ub_range_of_constant=UB_range_of_constant,
            min_chromosome_length=min_chromosome_length,     
            max_chromosome_length=max_chromosome_length,   

            # --- 3. Operator Specifics ---
            tournament_size=tournament_size,                             
            tournament_probability=tournament_probability,                    
            mutation_max_subtree_depth=random.randint(min_chromosome_length, max_chromosome_length),                
            homology_1_point_crossover_probability=homology_1_point_crossover_probability,    
            subtree_crossover_probability=subtree_crossover_probability,             
            uniform_crossover_probability=uniform_crossover_probability,             

            # --- 4. Problem Specific ---
            reference_point=optimal_cost,         
            solver_time_limit=SOLVER_TIME_LIMIT,
            
            # Optional: You can override available operations if needed
            available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"])

        # 6. Run EA
        start_time = pytime.time()
        with open(log_file_path, "w", encoding="utf-8") as f:
            with contextlib.redirect_stdout(f):
                best_ind = evolution_algorithm_execution(
                    didp_model_registry=creation_of_didp_model_function,
                    dual_bound_expression_function=dual_bound_expression_function,
                    params=params
                )
                print("\n" + "="*40 + "\nFINAL BEST INDIVIDUAL\n" + "="*40)
                print(best_ind)

        total_time = pytime.time() - start_time

        # 7. Save Result
        fitness_val = best_ind.get('fitness', -1) if best_ind else "FAILED"
        chrom_str = str(best_ind.get('chromosome', [])) if best_ind else "FAILED"

        result_data = {
            "Instance": instance_name,
            "Total_Time_(s)": round(total_time, 2),
            "Best_Fitness": fitness_val,
            "Best_Chromosome": chrom_str,
            "Log_File": log_file_path
        }
        
        file_exists = os.path.exists(OUTPUT_CSV)
        with open(OUTPUT_CSV, mode='a', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=result_data.keys())
            if not file_exists: writer.writeheader()
            writer.writerow(result_data)

        print(f"   ✅ Finished! Best Fit: {fitness_val} | Time: {total_time:.2f}s")
        processed_instances.add(instance_name)

    except Exception as e:
        print(f"   ❌ Failed: {e}")
        with open(log_file_path, "a") as f:
            f.write(f"\nCRITICAL ERROR: {e}")
            import traceback
            traceback.print_exc(file=f)
    finally:
        gc.collect()

print("\n🎉 Batch Run Complete!")

🧹 Starting Data Cleanup...
   -> Cleanup finished. Rows: 8 (was 8).
✨ Cleanup Complete.

🚀 MODE: BATCH RUN DETECTED (Source: CVRPTW_single_dual_bound_results_10s_lim.csv)
   -> Resuming: 8/10 instances already completed.

Processing R2_2_10.TXT (Ref Cost: 7238.458946841814)...
   ✅ Finished! Best Fit: 0.06295240547580538 | Time: 13763.46s

Processing RC2_2_10.TXT (Ref Cost: 5754.7715003469)...
   ✅ Finished! Best Fit: 0.04854873306490556 | Time: 13572.97s

🎉 Batch Run Complete!
